[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UoA-eResearch/embedding-llms-qualitative-data-workshop/blob/main/notebooks/01-environment-setup.ipynb)

# Notebook 01 — Environment Setup

**Duration:** ~20 minutes

Welcome. In this notebook you will set up the tools we use throughout the workshop.

By the end you will have:
- Python running in your browser
- A connection to a large language model (LLM) — an AI system that can read and generate text
- Enough confidence to keep going

You do not need to install anything on your computer. Everything runs here in **Google Colab** — a free coding environment that works entirely in your web browser. If you can use Google Docs, you can use Colab.

## What is a cell?

This notebook is made of **cells**. Each cell is a block that contains either text (like this one) or code (like the one below).

To run a code cell:
1. Click on it so it is highlighted
2. Press **Shift + Enter** on your keyboard (or click the ▶ play button on the left)

The result appears directly below the cell. If something goes wrong, you will see an error message — that is normal. Read the last line of the error. It usually tells you what happened.

Let's try it.

## Exercise 1 — Python as a calculator

Python can do arithmetic. Run the cell below to see it work.

In [ ]:
2 + 3

You should see the number `5` below the cell. That's it — you just ran Python.

Now try some of these. Change the numbers, or write your own. Run the cell each time to see the result.

In [ ]:
# Try any of these — change the numbers, or write your own.
# Delete the # at the start of a line to "uncomment" it and make it run.

# 10 * 5
# 100 / 3
# 2 ** 10        # ** means "to the power of"
# (4 + 6) * 2

Lines that start with `#` are **comments**. Python ignores them. They are notes for humans.

A cell only shows the result of the **last line**. If you want to see multiple results, use `print()` — we will learn that next.

## Exercise 2 — Variables and print()

A **variable** stores a value so you can use it later. Think of it as a labelled box: you put something in, and refer to it by name.

The `print()` function displays a value on screen. Run this cell to see both in action.

In [ ]:
name = "Kyle"
print("Hello,", name)
print("Welcome to the workshop.")

Your turn. Change `"____"` to your own name and run the cell.

In [ ]:
name = "____"    # <-- replace ____ with your name (keep the quotes)
print("Hello,", name)

# You can store numbers too:
sections = 200
print("The Privacy Act 2020 has roughly", sections, "sections.")
print("Reading them all would take a while. That is why we have an LLM.")

Notice: text goes in quotes (`"like this"`), numbers do not. If you see an error about quotes, check that both the opening and closing quotes are there.

## Where this fits in a real research workflow

> **Research context:** Everything we do today follows a workflow used in a published study that computationally analysed New Zealand's Mental Health Act (Ardekani et al., 2026). Step 1 of that workflow is **ingestion** — parsing the XML files that the NZ government publishes for every Act of Parliament. By the end of this notebook, you will have exactly the infrastructure needed for that step.

The six-step workflow from the paper is:

| Step | What it does | When we do it |
|------|-------------|---------------|
| 1. Ingestion | Parse raw data into a usable format | **This notebook** |
| 2. Extraction | Pull out structured information | Notebook 03 |
| 3. Expansion | Build outward from a starting point | Discussed in 03 |
| 4. Semantic enrichment | Use an LLM to summarise and clean | Notebook 04 |
| 5. Analysis | Find patterns and themes | Notebook 04 |
| 6. Interpretation | Check findings against reality | Notebooks 04 and 05 |

Keep this table in mind. We will come back to it at the end of the day.

## Setting up our tools

For the rest of the workshop we need a few tools:

| Tool | What it does |
|------|--------------|
| `groq` | Connects Python to an LLM (we use the free Groq API) |
| `requests` | Fetches data from the internet (we will download legislation) |
| `lxml` | Reads XML files — the format NZ legislation is published in |
| `Pillow` | Works with images (used in notebook 05) |

An **API** (Application Programming Interface) is a way for one program to talk to another over the internet. The Groq API lets our Python code send a question to an LLM and receive an answer back.

An **API key** is a password that identifies you to the service. You will create one in the next step.

## Get your Groq API key

Follow these steps. It takes about two minutes.

1. Go to [console.groq.com/keys](https://console.groq.com/keys)
2. Click **Sign in with Google** and use your university Google account
3. Once signed in, click **Create API Key**
4. Give it any name (e.g. "workshop")
5. **Copy the key** — it starts with `gsk_` and is a long string of letters and numbers
6. Paste it into the cell below, replacing `paste_your_key_here`

**Important:** the key only appears once. If you lose it, delete it and create a new one.

In [ ]:
# ============================================================
# SETUP CELL — Run this once at the start of every notebook
# ============================================================
# Paste your Groq API key between the quotes below, then run.

!pip install groq requests lxml Pillow
import os, json, base64, requests, io
from groq import Groq
from lxml import etree
from PIL import Image
from IPython.display import Image as IPImage, display

os.environ["GROQ_API_KEY"] = "paste_your_key_here"   # <-- paste your key here
client = Groq(api_key=os.environ["GROQ_API_KEY"])
TEXT_MODEL = "llama-3.3-70b-versatile"
VISION_MODEL = "meta-llama/llama-4-maverick-17b-128e-instruct"

print("Setup complete.")

You should see `Setup complete.` printed above, possibly after some installation messages. If you see an error, check:
- Did you paste the key between the quotes?
- Is the key complete? (It should be a long string starting with `gsk_`)
- Are there any extra spaces before or after the key?

If it works — congratulations. You now have a connection to an LLM from Python.

## Exercise 3 — Test your connection

Let's confirm everything works by sending a short message to the model and reading its reply.

This is the pattern you will use throughout the workshop:
1. Write a message
2. Send it to the model
3. Print the reply

Run the cell below. It asks the model a simple question.

In [ ]:
response = client.chat.completions.create(
    model=TEXT_MODEL,
    messages=[
        {"role": "user", "content": "In one sentence, what is the Privacy Act 2020?"}
    ]
)

print(response.choices[0].message.content)

You should see a one-sentence answer about New Zealand's Privacy Act. The exact wording may differ from your neighbour's — that is expected. We will explore why in the next notebook.

Let's break down what just happened:

| Line | What it does |
|------|--------------|
| `client.chat.completions.create(...)` | Sends a message to the LLM via the Groq API |
| `model=TEXT_MODEL` | Tells it which model to use (we set this in the setup cell) |
| `messages=[{...}]` | The conversation — a list of messages, each with a `role` and `content` |
| `"role": "user"` | This message is from you (the human) |
| `"content": "..."` | The actual text you are sending |
| `response.choices[0].message.content` | Extracts the model's reply from the response |

You will see this pattern in every notebook. The only thing that changes is the message you send.

### Your turn

Change the question below to anything you like and run the cell. Try asking it something related to your own research area.

In [ ]:
# Change the question to anything you like.
# Some ideas:
#   "What is thematic analysis in qualitative research?"
#   "Summarise the purpose of the NZ Human Rights Act 1993 in two sentences."
#   "What are the ethical risks of using AI in social science research?"

my_question = "What is thematic analysis in qualitative research?"

response = client.chat.completions.create(
    model=TEXT_MODEL,
    messages=[
        {"role": "user", "content": my_question}
    ]
)

print(response.choices[0].message.content)

## What you accomplished

In this notebook you:

- Ran Python code in your browser using Google Colab
- Learned what cells, variables, and `print()` do
- Installed the libraries we use throughout the workshop
- Created a Groq API key and connected to an LLM
- Sent your first message to the model and read its reply

This is the ingestion infrastructure — Step 1 of the research workflow we introduced earlier. Everything else builds on what you just set up.

**Next up:** In notebook 02, we look more closely at how the model behaves — why it sometimes gives different answers to the same question, and how the way you phrase a question changes what you get back.